# 🎮 Player Churn Analysis and Prediction

In this notebook, we explore the `online_gaming_behavior_dataset.csv`. 
Our goal is to understand what factors lead to a player being 'highly engaged' versus 'churning' (low engagement), and to build a predictive model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('online_gaming_behavior_dataset.csv')
df.head()

### Data Cleaning and Preprocessing
We will drop the `PlayerID` and map the target variable `EngagementLevel` to a binary Churn indicator. Low engagement means the player is churning.

In [ ]:
df = df.drop(columns=['PlayerID'])
df['Churn'] = df['EngagementLevel'].apply(lambda x: 1 if x == 'Low' else 0)
df = df.drop(columns=['EngagementLevel'])

### Exploratory Data Analysis (EDA)
Let's visualize some key factors that influence player churn.

In [ ]:
plt.figure(figsize=(15, 5))

# Churn rate by Genre
plt.subplot(1, 3, 1)
sns.barplot(data=df, x='GameGenre', y='Churn')
plt.title('Churn Rate by Game Genre')
plt.xticks(rotation=45)

# Churn rate by Game Difficulty
plt.subplot(1, 3, 2)
sns.barplot(data=df, x='GameDifficulty', y='Churn')
plt.title('Churn Rate by Game Difficulty')

# Churn rate by In-Game Purchases
plt.subplot(1, 3, 3)
sns.barplot(data=df, x='InGamePurchases', y='Churn')
plt.title('Churn Rate by In-Game Purchases')

plt.tight_layout()
plt.show()

# Additional churn summaries
churn_rate = df['Churn'].mean()
print(f"Overall churn rate: {churn_rate:.2%}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.countplot(data=df, x='Churn', ax=axes[0,0])
axes[0,0].set_title('Churn Count')

sns.barplot(data=df, x='Gender', y='Churn', ax=axes[0,1])
axes[0,1].set_title('Churn Rate by Gender')

sns.barplot(data=df, x='Location', y='Churn', ax=axes[1,0])
axes[1,0].set_title('Churn Rate by Location')
axes[1,0].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='PlayTimeHours', hue='Churn', element='step', stat='density', common_norm=False, ax=axes[1,1])
axes[1,1].set_title('Play Time Distribution by Churn')

plt.tight_layout()
plt.show()

Players with higher difficulty or lack of in-game purchases might have different churn patterns. Next, let's explore continuous variables like Play Time and Age.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x='Churn', y='PlayTimeHours', ax=axes[0])
axes[0].set_title('Play Time vs Churn')

sns.boxplot(data=df, x='Churn', y='Age', ax=axes[1])
axes[1].set_title('Age vs Churn')

plt.tight_layout()
plt.show()


Finally, let's look at the correlation among continuous variables and our target.

In [ ]:
numerical_features = ['Age', 'PlayTimeHours', 'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel', 'AchievementsUnlocked', 'Churn']
corr = df[numerical_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# pairwise scatter/hist relationships
sns.pairplot(df[numerical_features], hue='Churn', corner=True, diag_kind='hist')
plt.suptitle('Pairwise Relationships Among Numerical Features', y=1.02)
plt.show()

### Encoding Categorical Variables
We convert categorical features like Gender, Location, and Genre to numeric.

In [3]:
gender_map = {val: idx for idx, val in enumerate(df['Gender'].unique())}
df['Gender'] = df['Gender'].map(gender_map)

location_map = {val: idx for idx, val in enumerate(df['Location'].unique())}
df['Location'] = df['Location'].map(location_map)

genre_map = {val: idx for idx, val in enumerate(df['GameGenre'].unique())}
df['GameGenre'] = df['GameGenre'].map(genre_map)

diff_map = {val: idx for idx, val in enumerate(df['GameDifficulty'].unique())}
df['GameDifficulty'] = df['GameDifficulty'].map(diff_map)

### Model Training
Let's split the dataset, scale our features, and train a Random Forest Classifier.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

### Feature Importance
What features drive player churn the most?

In [5]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title('Feature Importances for Player Churn')
plt.show()

### Save Model and Scaler

In [6]:
import joblib
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Models saved successfully.')